## RUN - 3 - Correlation and similarity measurements

### Notes

- **Information on input**
    - `Ectoderm-data_clean_vec.pkl`; dictionary of Ectoderm data and vectors
    - `Myeloid-data_clean_vec.pkl`; dictionary of Myeloid data and vectors
    - Data structure: 
        - `'norm' vs 'dilu'` for normal vs diluted (1:1200) fibronectin conditions
            - `'Pos{ddd}'` for each explant sample
                - Pandas dfs of shape `tracks X (f, t, y, x, ...)*`
                - `*` Details on df columns:
                    - `f, t, y, x`: frames `[1]`, times `[min]`, positions `[microns]`
                    - `cen_y, cen_x, y_rel, x_rel, r, clust_r`: cluster center, relative positions, radial distances, cluster size `[microns]`
                    - `vy, vx, vy_itp, vx_itp`: velocities, locally interpolated ectoderm velocities `[microns/min]`
    - Tracking was done by Anh with StarDist for the ectoderm and manually in ImageJ for myeloid cells


* **Pre-requisites**
    - The data must have been preprocessed with `RUN - 1 - Preprocessing.ipynb`
    - Movement vectors must have been extracted and interpolated with `RUN - 2 - Movement vectors.ipynb`


- **Content of this notebook**
    1. Load the data
    2. Compute useful projections and derived measures from movement vectors
    3. Compute time-shift profiles of global/"population-wise" correlations between measured and interpolated vectors
        - Correlation of vector components (raw) [deprecated]
        - Correlation of vector components (normalized) [deprecated]
        - Correlation of vector magnitudes ("speed")
        - Correlation of x components ("para") of trajectory-aligned vectors ("trajectory speed")
        - Correlation of x components ("para") of trajectory-aligned & normalized vectors
        - Correlation of y components ("ortho") of trajectory-aligned vectors ("trajectory deviation")
        - Correlation of y components ("ortho") of trajectory-aligned & normalized vectors
        - Correlation of x components ("para") of ray-aligned vectors ("ray speed")
        - Correlation of x components ("para") of ray-aligned & normalized vectors
        - Correlation of y components ("ortho") of ray-aligned vectors ("ray deviation")
        - Correlation of y components ("ortho") of ray-aligned & normalized vectors
    4. Compute time-shift profiles of local/pairwise similarity metrics between measured and interpolated vectors
        - Cosine similarity ("angular similarity", "cos_sim")
        - Zscore similarity of vector magnitudes ("speed")
        - Zscore similarity of x components ("para") of trajectory-aligned vectors ("trajectory speed")
        - Zscore similarity of x components ("para") of trajectory-aligned & normalized vectors
        - Zscore similarity of y components ("ortho") of trajectory-aligned vectors ("trajectory deviation")
        - Zscore similarity of y components ("ortho") of trajectory-aligned & normalized vectors
        - Zscore similarity of x components ("para") of ray-aligned vectors ("ray speed")
        - Zscore similarity of x components ("para") of ray-aligned & normalized vectors
        - Zscore similarity of y components ("ortho") of ray-aligned vectors ("ray deviation")
        - Zscore similarity of y components ("ortho") of ray-aligned & normalized vectors
    5. Average correlation/similarity metrics over stable time span
    6. Save processed data for analysis


* **Outputs of this notebook**
    - _**Per-track main data**_
        - `ecto_data[conditions][positions]`; pandas dfs of shape `tracks X (f, t, y, x, ...)*`
        - `myel_data[conditions][positions]`; pandas dfs of shape `tracks X (f, t, y, x, ...)*`
        - `*` Details on df columns:
            - `f, t, y, x`: frames `[1]`, times `[min]`, positions `[microns]`
            - `cen_y, cen_x, y_rel, x_rel, r, clust_r`: cluster center, relative positions, radial distances, cluster size `[microns]`
            - `vy, vx, vy_itp, vx_itp`: velocities, locally interpolated ectoderm velocities `[microns/min]`
            - `v_mag, v_itp_mag`: velocity vector magnitudes
            - `vy_n, vx_n, vy_itp_n, vx_itp_n`: components of normalized (magnitude 1.0) vectors
            - `vy_traj, vx_traj, vy_traj_n, vx_traj_n`: cell track trajectory-aligned (optionally normalized) vectors
            - `vy_ray, vx_ray, vy_ray_n, vx_ray_n`: cluster radial axis-aligned (optionally normalized) vectors
    - _**Per-track similarity metrics**_
        - `ecto_similarities[conditions][positions]`; pandas dfs of shape `tracks X (f, ...)*`
        - `myel_similarities[conditions][positions]`; pandas dfs of shape `tracks X (f, ...)*`
        - `*` Details on df columns:
            - `f, t, y, x, cen_y, cen_x, y_rel, x_rel, r, clust_r`: as in `ecto_data` and `myel_data`
            - `metric + "-shift="+str(s) for s in profile_time_shifts`: Per-cell local similarity for each similarity metric
    - _**Time-shift profile data per frame**_
        - `ecto_profiles[conditions][positions][metric]`; pandas dfs of shape `f X profile_time_shifts`
        - `myel_profiles[conditions][positions][metric]`; pandas dfs of shape `f X profile_time_shifts`
    - _**Time-shift profile data averaged over frames**_
        - `ecto_profs_mean[conditions][positions][metric]`; pandas dfs of shape `profile_time_shifts`
        - `myel_profs_mean[conditions][positions][metric]`; pandas dfs of shape `profile_time_shifts`

### Prep

In [ ]:
### Imports

%load_ext autoreload
%autoreload 2

import os, warnings, pickle

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
from ipywidgets import interact

import sys; sys.path.insert(0, '..')
import tracking_analysis.profiles as proftools
from tracking_analysis.utilities import savebutton

In [ ]:
### Seeding

np.random.seed(42)

In [ ]:
### Parameters

# Overwrite or dry run
save_outputs = True

# Units
pxl_res  = 1.5152  # [microns]
time_res = 5       # [min]

# Shifted time profiles
prof_min_shift  = -30              # Range of profile into past
prof_max_shift  =  30              # Range of profile into future
prof_mean_range = pmr = (80, 180)  # Time point range for averaged profile

In [ ]:
### Data locations

top_path = r"..\Data\ex_vivo"

ecto_file = r"Ectoderm-data_clean_vec.pkl"
myel_file = r"Myeloid-data_clean_vec.pkl"

In [ ]:
### Load the data

# Ectoderm
with open(os.path.join(top_path, ecto_file), "rb") as infile:
    ecto_data = pickle.load(infile)

# Myeloid
with open(os.path.join(top_path, myel_file), "rb") as infile:
    myel_data = pickle.load(infile)

# Report
print("\nEctoderm data:")
display(ecto_data['norm'].keys())
display(ecto_data['dilu'].keys())
display(ecto_data['norm']['Pos001'].head())
display(ecto_data['norm']['Pos001'].shape)

print("\nMyeloid data:")
display(myel_data['norm'].keys())
display(myel_data['dilu'].keys())
display(myel_data['norm']['Pos001'].head())
display(myel_data['norm']['Pos001'].shape)

### Compute vector measures and projections

In [ ]:
### Compute vector magnitudes

# For each condition c and position p...
for c in ecto_data.keys():
    for p in ecto_data[c].keys():
        
        ecto_data[c][p]['v_mag']     = np.sqrt(np.sum(ecto_data[c][p][['vy', 'vx']].values**2.0, axis=1))
        ecto_data[c][p]['v_itp_mag'] = np.sqrt(np.sum(ecto_data[c][p][['vy_itp', 'vx_itp']].values**2.0, axis=1))
        myel_data[c][p]['v_mag']     = np.sqrt(np.sum(myel_data[c][p][['vy', 'vx']].values**2.0, axis=1))
        myel_data[c][p]['v_itp_mag'] = np.sqrt(np.sum(myel_data[c][p][['vy_itp', 'vx_itp']].values**2.0, axis=1))

# Show example
display(ecto_data[c][p].head())
display(myel_data[c][p].head())

In [ ]:
### Compute normalized vectors (magnitude=1.0)

# For each condition c and position p...
for c in ecto_data.keys():
    for p in ecto_data[c].keys():
        
        ecto_data[c][p][['vy_n', 'vx_n']]         = proftools.norm_vector(ecto_data[c][p][['vy', 'vx']])
        ecto_data[c][p][['vy_itp_n', 'vx_itp_n']] = proftools.norm_vector(ecto_data[c][p][['vy_itp', 'vx_itp']])
        myel_data[c][p][['vy_n', 'vx_n']]         = proftools.norm_vector(myel_data[c][p][['vy', 'vx']])
        myel_data[c][p][['vy_itp_n', 'vx_itp_n']] = proftools.norm_vector(myel_data[c][p][['vy_itp', 'vx_itp']])
        
# Show example
display(ecto_data[c][p].head())
display(myel_data[c][p].head())

In [ ]:
### Compute vectors aligned to start-end trajectory of each track

# For each condition c and position p...
for c in ecto_data.keys():
    for p in ecto_data[c].keys():
        
        # Point to the relevant data
        ecto_df = ecto_data[c][p]
        myel_df = myel_data[c][p]

        # Apply trajectory alignment function to ecto & myel for both v and v_itp
        ecto_df = proftools.align_to_start_end_trajectory(ecto_df, ['y', 'x'], ['vy', 'vx'])
        ecto_df = proftools.align_to_start_end_trajectory(ecto_df, ['y', 'x'], ['vy_itp', 'vx_itp'])
        myel_df = proftools.align_to_start_end_trajectory(myel_df, ['y', 'x'], ['vy', 'vx'])
        myel_df = proftools.align_to_start_end_trajectory(myel_df, ['y', 'x'], ['vy_itp', 'vx_itp'])

        # Normalize trajectory-aligned vectors (magnitude=1.0)
        ecto_df[['vy_traj_n', 'vx_traj_n']]         = proftools.norm_vector(ecto_df[['vy_traj', 'vx_traj']])
        ecto_df[['vy_itp_traj_n', 'vx_itp_traj_n']] = proftools.norm_vector(ecto_df[['vy_itp_traj', 'vx_itp_traj']])
        myel_df[['vy_traj_n', 'vx_traj_n']]         = proftools.norm_vector(myel_df[['vy_traj', 'vx_traj']])
        myel_df[['vy_itp_traj_n', 'vx_itp_traj_n']] = proftools.norm_vector(myel_df[['vy_itp_traj', 'vx_itp_traj']])

# Show example
display(ecto_data[c][p].head())
display(myel_data[c][p].head())

In [ ]:
### Visualize the trajectory-aligned vectors

@interact(condition=ecto_data.keys())
def show_by_condition(condition='norm'):
    @interact(position=ecto_data[condition].keys())
    @savebutton
    def show_by_position(position=list(ecto_data[condition].keys())[0]):
        
        # Point to the relevant data
        ecto_df = ecto_data[condition][position]
        myel_df = myel_data[condition][position]
        
        fig, ax = plt.subplots(2, 2, sharex=True, sharey=True, figsize=(5, 4.3))
        
        for i in np.random.randint(0, ecto_df.shape[0], size=500):
            ax[0,0].plot([0, ecto_df['vx'].values[i]], [0, ecto_df['vy'].values[i]], alpha=0.4)
            ax[0,1].plot([0, ecto_df['vx_traj'].values[i]], [0, ecto_df['vy_traj'].values[i]], alpha=0.4)
        for i in np.random.randint(0, myel_df.shape[0], size=500):
            ax[1,0].plot([0, myel_df['vx'].values[i]], [0, myel_df['vy'].values[i]], alpha=0.4)
            ax[1,1].plot([0, myel_df['vx_traj'].values[i]], [0, myel_df['vy_traj'].values[i]], alpha=0.4)
        
        for axis in ax.ravel():
            axis.set_aspect('equal', 'box')
        
        ax[0,0].set_title(f'{condition} - {position}')
        ax[0,1].set_title(f'{condition} - {position}')
        ax[0,0].set_ylabel("Ectoderm")
        ax[1,0].set_ylabel("Myeloid")
        ax[1,0].set_xlabel("Raw")
        ax[1,1].set_xlabel("Trajectory-aligned")
        
        plt.tight_layout()

In [ ]:
### Compute vectors aligned to the "cluster center --> cell position" ray

# For each condition c and position p...
for c in ecto_data.keys():
    for p in ecto_data[c].keys():
        
        # Point to the relevant data
        ecto_df = ecto_data[c][p]
        myel_df = myel_data[c][p]

        # Apply function to ecto & myel for both v and v_itp
        ecto_df = proftools.align_to_cluster_center_ray(ecto_df, ['y', 'x'], ['cen_y', 'cen_x'], ['vy', 'vx'])
        ecto_df = proftools.align_to_cluster_center_ray(ecto_df, ['y', 'x'], ['cen_y', 'cen_x'], ['vy_itp', 'vx_itp'])
        myel_df = proftools.align_to_cluster_center_ray(myel_df, ['y', 'x'], ['cen_y', 'cen_x'], ['vy', 'vx'])
        myel_df = proftools.align_to_cluster_center_ray(myel_df, ['y', 'x'], ['cen_y', 'cen_x'], ['vy_itp', 'vx_itp'])

        # Normalize ray-aligned vectors (magnitude=1.0)
        ecto_df[['vy_ray_n', 'vx_ray_n']]         = proftools.norm_vector(ecto_df[['vy_ray', 'vx_ray']])
        ecto_df[['vy_itp_ray_n', 'vx_itp_ray_n']] = proftools.norm_vector(ecto_df[['vy_itp_ray', 'vx_itp_ray']])
        myel_df[['vy_ray_n', 'vx_ray_n']]         = proftools.norm_vector(myel_df[['vy_ray', 'vx_ray']])
        myel_df[['vy_itp_ray_n', 'vx_itp_ray_n']] = proftools.norm_vector(myel_df[['vy_itp_ray', 'vx_itp_ray']])

# Show example
display(ecto_data[c][p].head())
display(myel_data[c][p].head())

In [ ]:
### Visualize the ray-aligned vectors

@interact(condition=ecto_data.keys())
def show_by_condition(condition='norm'):
    @interact(position=ecto_data[condition].keys())
    @savebutton
    def show_by_position(position=list(ecto_data[condition].keys())[0]):
        
        # Point to the relevant data
        ecto_df = ecto_data[condition][position]
        myel_df = myel_data[condition][position]
        
        fig, ax = plt.subplots(2, 2, sharex=True, sharey=True, figsize=(5, 4.3))
        
        for i in np.random.randint(0, ecto_df.shape[0], size=500):
            ax[0,0].plot([0, ecto_df['vx'].values[i]], [0, ecto_df['vy'].values[i]], alpha=0.4)
            ax[0,1].plot([0, ecto_df['vx_ray'].values[i]], [0, ecto_df['vy_ray'].values[i]], alpha=0.4)
        for i in np.random.randint(0, myel_df.shape[0], size=500):
            ax[1,0].plot([0, myel_df['vx'].values[i]], [0, myel_df['vy'].values[i]], alpha=0.4)
            ax[1,1].plot([0, myel_df['vx_ray'].values[i]], [0, myel_df['vy_ray'].values[i]], alpha=0.4)
         
        for axis in ax.ravel():
            axis.set_aspect('equal', 'box')
        
        ax[0,0].set_title(f'{condition} - {position}')
        ax[0,1].set_title(f'{condition} - {position}')
        ax[0,0].set_ylabel("Ectoderm")
        ax[1,0].set_ylabel("Myeloid")
        ax[1,0].set_xlabel("Raw")
        ax[1,1].set_xlabel("Ray-aligned")
        
        plt.tight_layout()

### Compute correlation/similarity profiles

In [ ]:
### Compute correlation profiles over time-shift windows

# Warning: Runtime is ~2h!

# Note: There are some edge cases emitting "RuntimeWarning: invalid value encountered in divide"
#       because all values in a vector set are the same (usually true zeros); these are expected
#       and the resulting nans are dealt with in the same as those from cases with insufficient
#       numbers of samples.

# Prep result dicts
ecto_profiles = {c : {p : {} for p in ecto_data[c].keys()} for c in ecto_data.keys()}
myel_profiles = {c : {p : {} for p in ecto_data[c].keys()} for c in ecto_data.keys()}

# For each condition c and position p...
for c in ecto_data.keys():
    for p in ecto_data[c].keys():

        # Point to the relevant data
        ecto_df = ecto_data[c][p].copy()
        myel_df = myel_data[c][p].copy()

        # Point to relevant results dict
        ecto_profs = ecto_profiles[c][p]
        myel_profs = myel_profiles[c][p]
        
        # Correlation of vector components (raw) [LEGACY; not a good measure!]
        ecto_profs['corr'] = proftools.col_correlation_profile(
            ecto_df, ['vy', 'vx'], ['vy_itp', 'vx_itp'], 
            prof_min_shift, prof_max_shift)
        myel_profs['corr'] = proftools.col_correlation_profile(
            myel_df, ['vy', 'vx'], ['vy_itp', 'vx_itp'], 
            prof_min_shift, prof_max_shift)

        # Correlation of normalized vector components [LEGACY; not a good measure!]
        ecto_profs['corr_n'] = proftools.col_correlation_profile(
            ecto_df, ('vx_n', 'vy_n'), ('vx_itp_n', 'vy_itp_n'), 
            prof_min_shift, prof_max_shift)
        myel_profs['corr_n'] = proftools.col_correlation_profile(
            myel_df, ('vx_n', 'vy_n'), ('vx_itp_n', 'vy_itp_n'), 
            prof_min_shift, prof_max_shift)
        
        # Correlation of vector magnitudes ("speed")
        ecto_profs['speed_corr'] = proftools.col_correlation_profile(
            ecto_df, 'v_mag', 'v_itp_mag', 
            prof_min_shift, prof_max_shift)
        myel_profs['speed_corr'] = proftools.col_correlation_profile(
            myel_df, 'v_mag', 'v_itp_mag', 
            prof_min_shift, prof_max_shift)
        
        # Correlation of x components ("para") of trajectory-aligned vectors ("trajectory speed")
        ecto_profs['traj_para_corr'] = proftools.col_correlation_profile(
            ecto_df, 'vx_traj', 'vx_itp_traj', 
            prof_min_shift, prof_max_shift)
        myel_profs['traj_para_corr'] = proftools.col_correlation_profile(
            myel_df, 'vx_traj', 'vx_itp_traj', 
            prof_min_shift, prof_max_shift)
        
        # Correlation of x components ("para") of trajectory-aligned & normalized vectors
        ecto_profs['traj_para_n_corr'] = proftools.col_correlation_profile(
            ecto_df, 'vx_traj_n', 'vx_itp_traj_n',
            prof_min_shift, prof_max_shift)
        myel_profs['traj_para_n_corr'] = proftools.col_correlation_profile(
            myel_df, 'vx_traj_n', 'vx_itp_traj_n', 
            prof_min_shift, prof_max_shift)
        
        # Correlation of y components ("ortho") of trajectory-aligned vectors ("trajectory deviation")
        ecto_profs['traj_ortho_corr'] = proftools.col_correlation_profile(
            ecto_df, 'vy_traj', 'vy_itp_traj',
            prof_min_shift, prof_max_shift)
        myel_profs['traj_ortho_corr'] = proftools.col_correlation_profile(
            myel_df, 'vy_traj', 'vy_itp_traj',
            prof_min_shift, prof_max_shift)
        
        # Correlation of y components ("ortho") of trajectory-aligned & normalized vectors
        ecto_profs['traj_ortho_n_corr'] = proftools.col_correlation_profile(
            ecto_df, 'vy_traj_n', 'vy_itp_traj_n',
            prof_min_shift, prof_max_shift)
        myel_profs['traj_ortho_n_corr'] = proftools.col_correlation_profile(
            myel_df, 'vy_traj_n', 'vy_itp_traj_n',
            prof_min_shift, prof_max_shift)
        
        # Correlation of x components ("para") of ray-aligned vectors ("ray speed")
        ecto_profs['ray_para_corr'] = proftools.col_correlation_profile(
            ecto_df, 'vx_ray', 'vx_itp_ray', 
            prof_min_shift, prof_max_shift)
        myel_profs['ray_para_corr'] = proftools.col_correlation_profile(
            myel_df, 'vx_ray', 'vx_itp_ray', 
            prof_min_shift, prof_max_shift)
        
        # Correlation of x components ("para") of ray-aligned & normalized vectors
        ecto_profs['ray_para_n_corr'] = proftools.col_correlation_profile(
            ecto_df, 'vx_ray_n', 'vx_itp_ray_n',
            prof_min_shift, prof_max_shift)
        myel_profs['ray_para_n_corr'] = proftools.col_correlation_profile(
            myel_df, 'vx_ray_n', 'vx_itp_ray_n',
            prof_min_shift, prof_max_shift)
        
        # Correlation of y components ("ortho") of ray-aligned vectors ("ray deviation")
        ecto_profs['ray_ortho_corr'] = proftools.col_correlation_profile(
            ecto_df, 'vy_ray', 'vy_itp_ray',
            prof_min_shift, prof_max_shift)
        myel_profs['ray_ortho_corr'] = proftools.col_correlation_profile(
            myel_df, 'vy_ray', 'vy_itp_ray',
            prof_min_shift, prof_max_shift)
        
        # Correlation of y components ("ortho") of ray-aligned & normalized vectors
        ecto_profs['ray_ortho_n_corr'] = proftools.col_correlation_profile(
            ecto_df, 'vy_ray_n', 'vy_itp_ray_n',
            prof_min_shift, prof_max_shift)
        myel_profs['ray_ortho_n_corr'] = proftools.col_correlation_profile(
            myel_df, 'vy_ray_n', 'vy_itp_ray_n',
            prof_min_shift, prof_max_shift)

In [ ]:
### Compute similarity profiles over time-shift windows (+keep local similarities)

# Warning: Runtime is ~6h!

# Prep additional results dict to keep local similarities
# Note: This is very elaborate for performance / memory efficiency reasons...
ref_cols = ['f', 't', 'y', 'x', 'cen_y', 'cen_x', 'y_rel', 'x_rel', 'r', 'clust_r']
metrics =  [
    'cos_sim', 'speed_sim', 
    'traj_para_sim', 'traj_para_n_sim', 'traj_ortho_sim', 'traj_ortho_n_sim',
    'ray_para_sim', 'ray_para_n_sim', 'ray_ortho_sim', 'ray_ortho_n_sim'
]
ecto_similarities, myel_similarities = {}, {}
for c in ecto_data.keys():
    ecto_similarities[c], myel_similarities[c] = {}, {}
    for p in ecto_data[c].keys():
        
        ecto_df_construction = [ecto_data[c][p][ref_cols].copy()]
        myel_df_construction = [myel_data[c][p][ref_cols].copy()]     
        for metric in metrics:
            ecto_df_construction.append(pd.DataFrame(
                np.empty((ecto_data[c][p].shape[0], -prof_min_shift+prof_max_shift+1), dtype=np.float32),
                index = ecto_data[c][p].index,
                columns = [metric+"-shift="+str(s) for s in range(prof_min_shift, prof_max_shift+1)]))
            myel_df_construction.append(pd.DataFrame(
                np.empty((myel_data[c][p].shape[0], -prof_min_shift+prof_max_shift+1), dtype=np.float32),
                index = myel_data[c][p].index,
                columns = [metric+"-shift="+str(s) for s in range(prof_min_shift, prof_max_shift+1)]))
        
        ecto_similarities[c][p] = pd.concat(ecto_df_construction, axis=1, copy=True)
        myel_similarities[c][p] = pd.concat(myel_df_construction, axis=1, copy=True)
        del ecto_df_construction
        del myel_df_construction

# For each condition c and position p...
for c in ecto_data.keys():
    for p in ecto_data[c].keys():

        # Point to the relevant data
        ecto_df = ecto_data[c][p].copy()
        myel_df = myel_data[c][p].copy()

        # Point to relevant results dicts
        ecto_profs = ecto_profiles[c][p]
        myel_profs = myel_profiles[c][p]
        ecto_sims = ecto_similarities[c][p]
        myel_sims = myel_similarities[c][p]
        
        # Cosine similarity ("angular similarity")
        ecto_sim_df, ecto_profs['cos_sim'] = proftools.vec_cosinesim_profile(
            ecto_df, ['vy', 'vx'], ['vy_itp', 'vx_itp'], 
            prof_min_shift, prof_max_shift)
        myel_sim_df, myel_profs['cos_sim'] = proftools.vec_cosinesim_profile(
            myel_df, ['vy', 'vx'], ['vy_itp', 'vx_itp'], 
            prof_min_shift, prof_max_shift)
        
        # Keep local similarities
        sims_columns = ["cos_sim-shift="+str(s) for s in range(prof_min_shift, prof_max_shift+1)]
        ecto_sims[sims_columns] = ecto_sim_df.astype(np.float32)
        myel_sims[sims_columns] = myel_sim_df.astype(np.float32)
        
        # Zscore similarity of vector magnitudes ("speed")
        ecto_sim_df, ecto_profs["speed_sim"] = proftools.col_similarity_profile(
            ecto_df, 'v_mag', 'v_itp_mag', 
            prof_min_shift, prof_max_shift)
        myel_sim_df, myel_profs["speed_sim"] = proftools.col_similarity_profile(
            myel_df, 'v_mag', 'v_itp_mag', 
            prof_min_shift, prof_max_shift)
        
        # Keep local similarities
        sims_columns = ["speed_sim-shift="+str(s) for s in range(prof_min_shift, prof_max_shift+1)]
        ecto_sims[sims_columns] = ecto_sim_df.astype(np.float32)
        myel_sims[sims_columns] = myel_sim_df.astype(np.float32)     
        
        # Zscore similarity of x components ("para") of trajectory-aligned vectors ("trajectory speed")
        ecto_sim_df, ecto_profs["traj_para_sim"] = proftools.col_similarity_profile(
            ecto_df, 'vx_traj', 'vx_itp_traj',
            prof_min_shift, prof_max_shift)
        myel_sim_df, myel_profs["traj_para_sim"] = proftools.col_similarity_profile(
            myel_df, 'vx_traj', 'vx_itp_traj',
            prof_min_shift, prof_max_shift)
        
        # Keep local similarities
        sims_columns = ["traj_para_sim-shift="+str(s) for s in range(prof_min_shift, prof_max_shift+1)]
        ecto_sims[sims_columns] = ecto_sim_df.astype(np.float32)
        myel_sims[sims_columns] = myel_sim_df.astype(np.float32)
        
        # Zscore similarity of x components ("para") of trajectory-aligned & normalized vectors
        ecto_sim_df, ecto_profs["traj_para_n_sim"] = proftools.col_similarity_profile(
            ecto_df, 'vx_traj_n', 'vx_itp_traj_n',
            prof_min_shift, prof_max_shift)
        myel_sim_df, myel_profs["traj_para_n_sim"] = proftools.col_similarity_profile(
            myel_df, 'vx_traj_n', 'vx_itp_traj_n',
            prof_min_shift, prof_max_shift)
        
        # Keep local similarities
        sims_columns = ["traj_para_n_sim-shift="+str(s) for s in range(prof_min_shift, prof_max_shift+1)]
        ecto_sims[sims_columns] = ecto_sim_df.astype(np.float32)
        myel_sims[sims_columns] = myel_sim_df.astype(np.float32)     
        
        # Zscore similarity of y components ("ortho") of trajectory-aligned vectors ("trajectory deviation")
        ecto_sim_df, ecto_profs["traj_ortho_sim"] = proftools.col_similarity_profile(
            ecto_df, 'vy_traj', 'vy_itp_traj',
            prof_min_shift, prof_max_shift)
        myel_sim_df, myel_profs["traj_ortho_sim"] = proftools.col_similarity_profile(
            myel_df, 'vy_traj', 'vy_itp_traj',
            prof_min_shift, prof_max_shift)
        
        # Keep local similarities
        sims_columns = ["traj_ortho_sim-shift="+str(s) for s in range(prof_min_shift, prof_max_shift+1)]
        ecto_sims[sims_columns] = ecto_sim_df.astype(np.float32)
        myel_sims[sims_columns] = myel_sim_df.astype(np.float32)
        
        # Zscore similarity of y components ("ortho") of trajectory-aligned & normalized vectors
        ecto_sim_df, ecto_profs["traj_ortho_n_sim"] = proftools.col_similarity_profile(
            ecto_df, 'vy_traj_n', 'vy_itp_traj_n',
            prof_min_shift, prof_max_shift)
        myel_sim_df, myel_profs["traj_ortho_n_sim"] = proftools.col_similarity_profile(
            myel_df, 'vy_traj_n', 'vy_itp_traj_n',
            prof_min_shift, prof_max_shift)
        
        # Keep local similarities
        sims_columns = ["traj_ortho_n_sim-shift="+str(s) for s in range(prof_min_shift, prof_max_shift+1)]
        ecto_sims[sims_columns] = ecto_sim_df.astype(np.float32)
        myel_sims[sims_columns] = myel_sim_df.astype(np.float32)
        
        # Zscore similarity of x components ("para") of ray-aligned vectors ("ray speed")
        ecto_sim_df, ecto_profs["ray_para_sim"] = proftools.col_similarity_profile(
            ecto_df, 'vx_ray', 'vx_itp_ray',
            prof_min_shift, prof_max_shift)
        myel_sim_df, myel_profs["ray_para_sim"] = proftools.col_similarity_profile(
            myel_df, 'vx_ray', 'vx_itp_ray',
            prof_min_shift, prof_max_shift)
        
        # Keep local similarities
        sims_columns = ["ray_para_sim-shift="+str(s) for s in range(prof_min_shift, prof_max_shift+1)]
        ecto_sims[sims_columns] = ecto_sim_df.astype(np.float32)
        myel_sims[sims_columns] = myel_sim_df.astype(np.float32)
        
        # Zscore similarity of x components ("para") of ray-aligned & normalized vectors
        ecto_sim_df, ecto_profs["ray_para_n_sim"] = proftools.col_similarity_profile(
            ecto_df, 'vx_ray_n', 'vx_itp_ray_n',
            prof_min_shift, prof_max_shift)
        myel_sim_df, myel_profs["ray_para_n_sim"] = proftools.col_similarity_profile(
            myel_df, 'vx_ray_n', 'vx_itp_ray_n',
            prof_min_shift, prof_max_shift)
        
        # Keep local similarities
        sims_columns = ["ray_para_n_sim-shift="+str(s) for s in range(prof_min_shift, prof_max_shift+1)]
        ecto_sims[sims_columns] = ecto_sim_df.astype(np.float32)
        myel_sims[sims_columns] = myel_sim_df.astype(np.float32)
        
        # Zscore similarity of y components ("ortho") of ray-aligned vectors ("ray deviation")
        ecto_sim_df, ecto_profs["ray_ortho_sim"] = proftools.col_similarity_profile(
            ecto_df, 'vy_ray', 'vy_itp_ray',
            prof_min_shift, prof_max_shift)
        myel_sim_df, myel_profs["ray_ortho_sim"] = proftools.col_similarity_profile(
            myel_df, 'vy_ray', 'vy_itp_ray',
            prof_min_shift, prof_max_shift)
        
        # Keep local similarities
        sims_columns = ["ray_ortho_sim-shift="+str(s) for s in range(prof_min_shift, prof_max_shift+1)]
        ecto_sims[sims_columns] = ecto_sim_df.astype(np.float32)
        myel_sims[sims_columns] = myel_sim_df.astype(np.float32)
        
        # Zscore similarity of y components ("ortho") of ray-aligned & normalized vectors
        ecto_sim_df, ecto_profs["ray_ortho_n_sim"] = proftools.col_similarity_profile(
            ecto_df, 'vy_ray_n', 'vy_itp_ray_n',
            prof_min_shift, prof_max_shift)
        myel_sim_df, myel_profs["ray_ortho_n_sim"] = proftools.col_similarity_profile(
            myel_df, 'vy_ray_n', 'vy_itp_ray_n',
            prof_min_shift, prof_max_shift)
        
        # Keep local similarities
        sims_columns = ["ray_ortho_n_sim-shift="+str(s) for s in range(prof_min_shift, prof_max_shift+1)]
        ecto_sims[sims_columns] = ecto_sim_df.astype(np.float32)
        myel_sims[sims_columns] = myel_sim_df.astype(np.float32)

In [ ]:
### Visualize the resulting correlation/similarity profiles

@interact(condition=ecto_data.keys())
def show_by_condition(condition='norm'):
    @interact(position=ecto_data[condition].keys())
    @savebutton
    def show_by_position(position=list(ecto_data[condition].keys())[0]):
        
        # Select relevant data
        ecto_profs = ecto_profiles[condition][position]
        myel_profs = myel_profiles[condition][position]
        
        # Prep
        fig, ax = plt.subplots(
            2, len(ecto_profs.keys()), 
            figsize=(1.1*len(ecto_profs.keys()), 6), 
            sharex=True, sharey=True)

        # For each metric...
        for m, metric in enumerate(ecto_profs.keys()):

            # Generate the heatmap
            ax[0,m].imshow(ecto_profs[metric], interpolation='none')
            ax[1,m].imshow(myel_profs[metric], interpolation='none')

            # Set cosmetics
            ax[0,m].set_title(metric, fontsize=9.5)
            ax[1,m].set_xlabel('$shift$')
            ax[1,m].set_xticks([0, (prof_max_shift-prof_min_shift)//2, prof_max_shift-prof_min_shift])
            ax[1,m].set_xticklabels([prof_min_shift, prof_min_shift+(prof_max_shift-prof_min_shift)//2, prof_max_shift])

        # More cosmetics
        ax[0,0].set_ylabel('Ectoderm\n\n$frame$')
        ax[1,0].set_ylabel('Myeloid\n\n$frame$')

        # Finalize
        plt.tight_layout()

In [ ]:
### Average metrics over stable time range to get robust profiles

# Prep result dicts
ecto_profs_mean = {c : {p : {} for p in ecto_data[c].keys()} for c in ecto_data.keys()}
myel_profs_mean = {c : {p : {} for p in ecto_data[c].keys()} for c in ecto_data.keys()}

# For each condition c and position p...
for c in ecto_data.keys():
    for p in ecto_data[c].keys():
        
        # Get mask of frames within prof_mean_range (pmr)
        metrics = m = list(ecto_profiles[c][p].keys())[0]
        ecto_mean_mask = (ecto_profiles[c][p][m].index >= pmr[0]) & (ecto_profiles[c][p][m].index <= pmr[1])
        myel_mean_mask = (myel_profiles[c][p][m].index >= pmr[0]) & (myel_profiles[c][p][m].index <= pmr[1])
        
        # For each metric...
        for metric in ecto_profiles[c][p].keys():
            
            # Compute average within the masks
            ecto_profs_mean[c][p][metric] = ecto_profiles[c][p][metric].loc[ecto_mean_mask].mean(axis=0)
            myel_profs_mean[c][p][metric] = myel_profiles[c][p][metric].loc[myel_mean_mask].mean(axis=0)

In [ ]:
### Visualize resulting averaged correlation/similarity profiles

@interact(condition=ecto_data.keys())
def show_by_condition(condition='norm'):
    @interact(position=ecto_data[condition].keys(),
              yrange=["indiv", "common"])
    @savebutton
    def show_by_position(position=list(ecto_data[condition].keys())[0],
                         yrange="indiv"):
        
        # Select relevant data
        ecto_pms = ecto_profs_mean[condition][position]
        myel_pms = myel_profs_mean[condition][position]
        
        # Prep
        fig, ax = plt.subplots(
            int(np.ceil(len(ecto_pms.keys())/2)), 2, 
            figsize=(8, 2*int(np.ceil(len(ecto_pms.keys())/2))), 
            sharex=True, sharey=yrange=="common")
        if len(ecto_pms.keys()) % 2 != 0:
            ax[-1, -1].set_visible(False)

        # For each metric...
        for m, metric in enumerate(ecto_pms.keys()):

            # Plot profiles
            ax[m//2, m%2].plot(
                ecto_pms[metric].index * time_res, 
                ecto_pms[metric], 
                label='Ectoderm')
            ax[m//2, m%2].plot(
                myel_pms[metric].index * time_res, 
                myel_pms[metric], 
                label='Myeloid')

            # Set cosmetics
            ax[m//2, m%2].set_title(f"{condition} - {position} - {metric}")
            if (m//2) == 2:
                ax[m//2, m%2].set_xlabel('time shift [min]')
            if (m%2) == 0:
                ax[m//2, m%2].set_ylabel('metric')

        # Axis settings
        plt.xlim(-100, 100)     # Crop a bit to visualize peaks better
        if yrange=="common":    # Set natural range
            plt.ylim(0.0, 1.0)  

        # Add midlines...
        for m in range(ax.size):
            ymin, ymax = ax[m//2, m%2].get_ylim()
            ax[m//2, m%2].vlines(0, ymin, ymax, color='k', lw=0.5, alpha=0.3, zorder=-1)
            ax[m//2, m%2].set_ylim(ymin, ymax)

        # Finalize
        plt.tight_layout()

### Save the results

In [ ]:
### if save_outputs:

    # Ectoderm
    with open(os.path.join(top_path, "Ectoderm-data_clean_vec_prof.pkl"), "wb") as outfile:
        pickle.dump(ecto_data, outfile, protocol=pickle.HIGHEST_PROTOCOL)
    with open(os.path.join(top_path, "Ectoderm-profiles.pkl"), "wb") as outfile:
        pickle.dump(ecto_profiles, outfile, protocol=pickle.HIGHEST_PROTOCOL)
    with open(os.path.join(top_path, "Ectoderm-similarities.pkl"), "wb") as outfile:
        pickle.dump(ecto_similarities, outfile, protocol=pickle.HIGHEST_PROTOCOL)
    with open(os.path.join(top_path, "Ectoderm-profiles_mean.pkl"), "wb") as outfile:
        pickle.dump(ecto_profs_mean, outfile, protocol=pickle.HIGHEST_PROTOCOL)

    # Myeloid
    with open(os.path.join(top_path, "Myeloid-data_clean_vec_prof.pkl"), "wb") as outfile:
        pickle.dump(myel_data, outfile, protocol=pickle.HIGHEST_PROTOCOL)
    with open(os.path.join(top_path, "Myeloid-profiles.pkl"), "wb") as outfile:
        pickle.dump(myel_profiles, outfile, protocol=pickle.HIGHEST_PROTOCOL)
    with open(os.path.join(top_path, "Myeloid-similarities.pkl"), "wb") as outfile:
        pickle.dump(myel_similarities, outfile, protocol=pickle.HIGHEST_PROTOCOL)
    with open(os.path.join(top_path, "Myeloid-profiles_mean.pkl"), "wb") as outfile:
        pickle.dump(myel_profs_mean, outfile, protocol=pickle.HIGHEST_PROTOCOL)